In [28]:
import os
import wandb # для логирования

import numpy as np
import random
from tqdm import *
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim # для оптимизаторов
from torchvision import datasets # для данных
import torchvision.transforms as transforms # для преобразований тензоров
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt

In [29]:
df= pd.read_csv('/Users/phuongnguyen/Downloads/car_policy.csv')

df.head()

,policy_tenure,age_of_car,age_of_policyholder,population_density,make,max_torque,max_power,airbags,is_esc,is_adjustable_steering,...,engine_type_K Series Dual jet,engine_type_K10C,engine_type_i-DTEC,rear_brakes_type_Drum,transmission_type_Manual,steering_type_Manual,steering_type_Power,safe_score,car_size,age_of_car_and_policy
0,0.515874,0.05,0.644231,4990,1,60.0,40.36,2,0,0,...,0,0,0,1,1,0,1,2,7698283125,0.025794
1,0.672619,0.02,0.375000,27003,1,60.0,40.36,2,0,0,...,0,0,0,1,1,0,1,2,7698283125,0.013452
2,0.841110,0.02,0.384615,4076,1,60.0,40.36,2,0,0,...,0,0,0,1,1,0,1,2,7698283125,0.016822
3,0.900277,0.11,0.432692,21622,1,113.0,88.50,2,1,1,...,0,0,0,1,0,0,0,6,10500957375,0.099030
4,0.596403,0.11,0.634615,34738,2,91.0,67.06,2,0,0,...,0,0,0,1,0,0,0,3,8777961010,0.065604


In [30]:
# Разделение на X и y
X = df.drop(columns = ['is_claim'])
y = df['is_claim']

print(X.shape)
print(y.shape)

(58592, 89)
(58592,)


In [31]:
y.value_counts()

is_claim
0    54844
1     3748
Name: count, dtype: int64

In [32]:
# Зафиксируем seed для воспроизводимости

def seed_everything(seed):
    random.seed(seed) # фиксируем генератор случайных чисел
    os.environ['PYTHONHASHSEED'] = str(seed) # фиксируем заполнения хешей
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU
    #torch.backends.cudnn.deterministic = True # выбираем только детерминированные алгоритмы (для сверток)
    #torch.backends.cudnn.benchmark = False # фиксируем алгоритм вычисления сверток

In [ ]:
class CFG:

# Задаем параметры нашего эксперимента

  api = "-------"# вписать свой API Wandb
  project = "Models"# вписать название эксперимента, который предварительно надо создать в Wandb
  num_epochs = 10 # количество эпох
  train_batch_size = 64 # размер батча обучающей выборки
  test_batch_size = 512 # размер батча тестовой выборки
  num_workers = 2 # количество активных процессов на загрузку данных
  lr = 0.001 # learning_rate
  seed = 2022 # для функции воспроизводимости
  wandb = True # флаг использования Wandb

In [34]:
 #Поделим данные на train, test

X_train, X_test,  y_train, y_test  = train_test_split(X, y, test_size= 0.2, random_state = CFG.seed, stratify = y)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


(46873, 89)
(11719, 89)
(46873,)
(11719,)


In [37]:
# Стандартизируем наги значения 
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#Превращаю в тензор
X_train_tens = torch.tensor(X_train_scaled, dtype  = torch.float32)
X_test_tens = torch.tensor(X_test_scaled, dtype  = torch.float32)
y_train_tens = torch.tensor(y_train.values, dtype  = torch.float32).reshape(-1, 1)
y_test_tens = torch.tensor(y_test.values, dtype  = torch.float32).reshape(-1, 1)


# Создаю train_dataset из X_train_tens и y_train_tens и  test_dataset из X_test_tens и y_test_tens
train_dataset = TensorDataset(X_train_tens, y_train_tens)
test_dataset = TensorDataset(X_test_tens, y_test_tens)

#создаю лоудары, чтобы передавались данные батчами
train_loader = DataLoader(train_dataset, batch_size= CFG.train_batch_size, shuffle= True, num_workers= CFG.num_workers)
test_loader = DataLoader(test_dataset, batch_size= CFG.test_batch_size, shuffle = False, num_workers= CFG.num_workers)




In [38]:
examples = enumerate(train_loader)

batch_ind, (example_data, example_targets ) = next(examples)

In [39]:
example_data.shape

torch.Size([64, 89])

## Первая модель